# Phase 1 数据探索

## Synthetic Data 声明

本 Notebook 只读分析程序生成的 **Synthetic E-commerce Data / 模拟电商数据**，不代表真实企业、真实用户或真实经营表现。Notebook 不构建、不修改、不覆盖任何数据快照。

## 数据范围

分析对象为五张业务表：`products`、`customers`、`traffic`、`marketing` 和 `orders`。下方代码会从当前工作目录向上定位项目根目录，避免依赖 Notebook 的启动位置。

若快照不存在，请先在项目根目录运行 `make PYTHON=python3.12 phase1-data`（也可将 `PYTHON` 替换为其他 Python 3.11+ 可执行文件），再执行本 Notebook。

In [ ]:
from pathlib import Path
import json

import pandas as pd
import yaml


def find_project_root() -> Path:
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "configs" / "data"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "未找到项目根目录；请从仓库内启动 Jupyter，并先运行 make phase1-data。"
    )


PROJECT_ROOT = find_project_root()
SNAPSHOTS = {
    "Development": PROJECT_ROOT / "data/synthetic/v1",
    "Holdout": PROJECT_ROOT / "data/synthetic/holdout-v1",
}
CONFIGS = {
    "Development": PROJECT_ROOT / "configs/data/synthetic_v1.yaml",
    "Holdout": PROJECT_ROOT / "configs/data/synthetic_holdout_v1.yaml",
}

missing = [name for name, path in SNAPSHOTS.items() if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        f"缺少数据快照 {missing}；请先在项目根目录运行 make phase1-data。"
    )

manifests = {
    name: json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    for name, path in SNAPSHOTS.items()
}
quality_reports = {
    name: json.loads(
        (path / "data_quality_report.json").read_text(encoding="utf-8")
    )
    for name, path in SNAPSHOTS.items()
}
configs = {
    name: yaml.safe_load(path.read_text(encoding="utf-8"))
    for name, path in CONFIGS.items()
}
tables = {
    dataset: {
        table: pd.read_parquet(snapshot / metadata["file"])
        for table, metadata in manifests[dataset]["tables"].items()
    }
    for dataset, snapshot in SNAPSHOTS.items()
}

In [ ]:
range_rows = []
for dataset, dataset_tables in tables.items():
    traffic = dataset_tables["traffic"]
    orders = dataset_tables["orders"]
    range_rows.append(
        {
            "dataset": dataset,
            "dataset_version": manifests[dataset]["dataset_version"],
            "dataset_id": manifests[dataset]["dataset_id"],
            "start_date": traffic["date"].min(),
            "end_date": traffic["date"].max(),
            "products": len(dataset_tables["products"]),
            "customers": len(dataset_tables["customers"]),
            "orders": orders["order_id"].nunique(),
        }
    )

pd.DataFrame(range_rows)

## 质量检查

展示每套冻结快照的质量状态、Schema 错误、失败检查与 Manifest 行数。这里仅读取已生成的质量报告，不在 Notebook 中复制生产校验逻辑。

In [ ]:
quality_rows = []
for dataset, report in quality_reports.items():
    quality_rows.append(
        {
            "dataset": dataset,
            "source_label": report["source_label"],
            "status": report["status"],
            "schema_error_count": len(report["schema_errors"]),
            "failed_check_count": len(report["failed_checks"]),
            **{f"rows_{name}": count for name, count in report["row_counts"].items()},
        }
    )

pd.DataFrame(quality_rows)

## 指标分布

按日聚合并描述业务指标分布：**GMV**、**Orders**、**AOV**、**CTR**、**CVR**、**Refund Rate** 和 **ROAS**。这些是模拟业务数据的描述统计，不是 Agent 评测结果。

In [ ]:
def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.div(denominator.where(denominator.ne(0)))


daily_metrics = {}
for dataset, dataset_tables in tables.items():
    orders = dataset_tables["orders"]
    traffic = dataset_tables["traffic"]
    marketing = dataset_tables["marketing"]

    daily_orders = orders.groupby("order_date").agg(
        GMV=("revenue", "sum"),
        Orders=("order_id", "nunique"),
        Refunds=("is_refund", "sum"),
    )
    daily_traffic = traffic.groupby("date").agg(
        Impressions=("impressions", "sum"),
        Clicks=("clicks", "sum"),
        Visits=("visits", "sum"),
    )
    daily_marketing = marketing.groupby("date").agg(Spend=("spend", "sum"))
    daily = daily_traffic.join(daily_orders, how="left").join(
        daily_marketing, how="left"
    )
    daily["AOV"] = safe_ratio(daily["GMV"], daily["Orders"])
    daily["CTR"] = safe_ratio(daily["Clicks"], daily["Impressions"])
    daily["CVR"] = safe_ratio(daily["Orders"], daily["Visits"])
    daily["Refund Rate"] = safe_ratio(daily["Refunds"], daily["Orders"])
    daily["ROAS"] = safe_ratio(daily["GMV"], daily["Spend"])
    daily_metrics[dataset] = daily[
        ["GMV", "Orders", "AOV", "CTR", "CVR", "Refund Rate", "ROAS"]
    ]

metric_distribution = pd.concat(daily_metrics, names=["dataset", "date"])
metric_distribution.groupby(level="dataset").describe().transpose()

## 异常场景

配置显式注入销量下降、流量下降、转化下降、高退款、流量缺失、极端流量峰值和多因素下降场景。下面并列展示两套配置中的场景定义，以及按商品汇总的可观察事实；配置用于解释场景范围，不作为 Agent 的 Gold 答案。

In [ ]:
anomaly_config = pd.concat(
    {
        dataset: pd.DataFrame(config["anomalies"])
        for dataset, config in configs.items()
    },
    names=["dataset", "row"],
).reset_index(level="row", drop=True)
anomaly_config

In [ ]:
observable_rows = []
for dataset, dataset_tables in tables.items():
    traffic = dataset_tables["traffic"]
    orders = dataset_tables["orders"]
    traffic_by_product = traffic.groupby("product_id", as_index=False).agg(
        visits=("visits", "sum"),
        missing_traffic_days=("is_missing", "sum"),
        max_impressions=("impressions", "max"),
    )
    orders_by_product = orders.groupby("product_id", as_index=False).agg(
        orders=("order_id", "nunique"),
        gmv=("revenue", "sum"),
        refund_rate=("is_refund", "mean"),
    )
    observed = traffic_by_product.merge(orders_by_product, on="product_id")
    observed.insert(0, "dataset", dataset)
    observable_rows.append(observed)

pd.concat(observable_rows, ignore_index=True).sort_values(
    ["dataset", "product_id"]
)

## 双数据集说明

- **Development** 使用 `data/synthetic/v1`，用于开发期分析与验证。
- **Holdout** 使用独立 Seed 生成的 `data/synthetic/holdout-v1`，用于最终评测隔离；开发过程中不得根据其 Gold 内容调优 Agent。
- 两套快照分别拥有 Dataset ID、配置哈希和文件哈希。Notebook 只做并列描述，不合并身份，也不把一套数据的结论外推到另一套。

In [ ]:
pd.DataFrame(
    [
        {
            "dataset": dataset,
            "dataset_version": manifest["dataset_version"],
            "dataset_id": manifest["dataset_id"],
            "seed": manifest["seed"],
            "config_sha256": manifest["config_sha256"],
            "generator_source_sha256": manifest["generator_source_sha256"],
        }
        for dataset, manifest in manifests.items()
    ]
)

## 局限性

- 数据由固定规则和 Seed 生成，只用于受控评测，不代表真实电商生产分布。
- 异常类型、强度、持续时间和实体覆盖由配置限定，不能覆盖开放世界中的全部异常。
- 描述统计只能说明当前快照中的模式，不能证明因果关系或生产泛化能力。
- Holdout 虽与 Development 独立生成，但仍共享同一生成器机制，分布差异有限。
- 本阶段未运行 Agent；所有 Agent Evaluation 结果均为 **Pending / Not Run**，因此这里不报告任何 Agent 效果数字。
- Notebook 不保存执行输出；查看结果时应在本地基于已验证快照重新运行。